# Mario PPO: notebook huấn luyện trên Kaggle (phiên CPU)

**Trước khi chạy:**
1. Settings → Accelerator = **None** (phiên CPU, không tốn quota GPU). Internet = **On** (cần tài khoản đã xác minh số điện thoại).
2. Nếu repo GitHub để private: Add-ons → Secrets → thêm `GITHUB_TOKEN` (token chỉ cần quyền đọc repo).
3. Nếu chạy tiếp lần trước: Add Input → chọn **output của phiên trước** (notebook này, phiên trước), rồi điền `RESUME_DIR` ở ô cấu hình.
4. Sửa ô cấu hình bên dưới, rồi **Save Version → Save & Run All (Commit)** để chạy không cần tương tác (tối đa 12 giờ).

Mọi lệnh Python chạy qua `!python ...` trong tiến trình mới, nên **không cần khởi động lại kernel** sau khi cài `numpy<2`.
Sau mỗi phiên, ghi một dòng vào bảng theo dõi chung (ô cuối in sẵn thông tin cần ghi).

In [ ]:
# ===== CẤU HÌNH LẦN CHẠY (chỉ sửa ô này) =====
NGUOI      = "Hien"                 # người chạy
REPO       = "github.com/studie-e/Mario-and-Peaches"
REF        = "v1-code"              # tag/nhánh/commit; sau Buổi 2 luôn dùng v1-code hoặc commit có wrapper mới
LEVEL      = "1-1"
RUN_NAME   = "gamma09-seed1"        # tên lần chạy, giữ NGUYÊN qua các phiên resume
SEED       = 1
OVERRIDES  = "device=cpu algo.total_steps=3000000 algo.gamma=0.9"   # khóa=giá trị, cách nhau bởi dấu cách
# Chạy tiếp: thư mục run của phiên trước trong /kaggle/input, ví dụ
# "/kaggle/input/mario-ppo-hien/runs/1-1/gamma09-seed1". Để None nếu là phiên đầu.
RESUME_DIR = None
# Học chuyển giao (phiên đầu ở màn mới): checkpoint đã qua scripts/chuyen_giao.py, ví dụ
# "/kaggle/input/xuat-phat-1-1/xuat-phat-1-1.pt". Bỏ qua nếu RESUME_DIR khác None.
INIT_CKPT  = None
TIME_LIMIT = "11h30m"               # dừng êm trước giới hạn 12 giờ để output kịp lưu
RUN_PYTEST = True

In [ ]:
!nproc && free -h && python --version

In [ ]:
import os, subprocess
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass  # repo public hoặc chưa thêm secret
url = f"https://{token}@{REPO}.git" if token else f"https://{REPO}.git"
subprocess.run(["rm", "-rf", "/tmp/repo"], check=True)
subprocess.run(["git", "clone", "-q", url, "/tmp/repo"], check=True)
subprocess.run(["git", "-C", "/tmp/repo", "checkout", "-q", REF], check=True)
COMMIT = subprocess.check_output(["git", "-C", "/tmp/repo", "rev-parse", "--short", "HEAD"], text=True).strip()
os.chdir("/tmp/repo")
print("commit", COMMIT)

In [ ]:
# Kaggle có sẵn torch, chỉ cài phần còn lại (nes-py cần biên dịch, mất vài phút)
!pip install -q -e .
!python -c "import numpy, torch; print('numpy', numpy.__version__, '| torch', torch.__version__)"

In [ ]:
if RUN_PYTEST:
    !python -m pytest -q

In [ ]:
import glob, shutil
RUN_DIR = f"/kaggle/working/runs/{LEVEL}/{RUN_NAME}"
resume = None
if RESUME_DIR:
    # chép cả thư mục run cũ để progress.csv/evals.csv được nối tiếp, rồi resume từ checkpoint mới nhất
    shutil.copytree(RESUME_DIR, RUN_DIR, dirs_exist_ok=True)
    ckpts = sorted(glob.glob(f"{RUN_DIR}/checkpoints/step_*.pt"))
    assert ckpts, f"không thấy checkpoint trong {RESUME_DIR}"
    resume = ckpts[-1]
elif INIT_CKPT:
    resume = INIT_CKPT
os.makedirs(RUN_DIR, exist_ok=True)
args = f"level={LEVEL} seed={SEED} run_dir={RUN_DIR} {OVERRIDES}" + (f" resume={resume}" if resume else "")
print("resume:", resume)
print("python -m mario_ppo.train", args)

In [ ]:
# SIGINT khi hết TIME_LIMIT: train.py đóng môi trường và tệp CSV trong khối finally;
# các checkpoint đã lưu (mỗi 500k bước) vẫn còn để phiên sau resume.
!timeout -s INT {TIME_LIMIT} python -m mario_ppo.train {args} 2>&1 | tee -a {RUN_DIR}/train.log

In [ ]:
import pandas as pd
prog = pd.read_csv(f"{RUN_DIR}/progress.csv")
print(prog.tail(5).to_string())
if os.path.exists(f"{RUN_DIR}/evals.csv"):
    print(pd.read_csv(f"{RUN_DIR}/evals.csv").to_string())
ckpts = sorted(glob.glob(f"{RUN_DIR}/checkpoints/step_*.pt"))
last = prog.iloc[-1]
print("\n===== GHI VÀO BẢNG THEO DÕI =====")
print(f"người: {NGUOI} | commit: {COMMIT} | màn: {LEVEL} | run: {RUN_NAME} | seed: {SEED}")
print(f"cấu hình: {OVERRIDES}")
print(f"bước: {int(last.global_step)} | bước/giây: {last.sps} | x_pos: {last.x_pos} | flag_rate(100 lượt): {last.flag_rate}")
print(f"checkpoint mới nhất: {ckpts[-1] if ckpts else '(chưa có)'}")